#Transfer Learning

🧠 What is Transfer Learning?

Transfer learning means taking a pre-trained model (one that has already learned from a large dataset) and reusing it for a new but related task — instead of training a model from scratch.

Example: Medical image analysis

⚙️ How It Works

Start with a pre-trained model (like ResNet, VGG, or BERT).

Freeze most of its layers (keep existing learned features).

Replace the final layer(s) with new ones suited to your target task.

Train (fine-tune) on your dataset.

In [ ]:
import tensorflow as tf

In [ ]:
import matplotlib.pylab as plt

import tensorflow_hub as hub
import tensorflow_datasets as tfds

from tensorflow.keras import layers

In [ ]:
import logging
logger = tf.get_logger()
logger.setLevel(logging.ERROR)

Use a TensorFlow Hub MobileNet for prediction

In [ ]:
CLASSIFIER_URL ="https://tfhub.dev/google/tf2-preview/mobilenet_v2/classification/2"
IMAGE_RES = 224

class MyModel(tf.keras.Model):
  def __init__(self, classifier_url, image_res):
    super(MyModel, self).__init__()
    self.classifier = hub.KerasLayer(classifier_url, input_shape=(image_res, image_res, 3))

  def call(self, inputs):
    return self.classifier(inputs)

model = MyModel(CLASSIFIER_URL, IMAGE_RES)

In [ ]:
import numpy as np
import PIL.Image as Image

grace_hopper = tf.keras.utils.get_file('image.jpg','https://storage.googleapis.com/download.tensorflow.org/example_images/grace_hopper.jpg')
grace_hopper = Image.open(grace_hopper).resize((IMAGE_RES, IMAGE_RES))
grace_hopper

In [ ]:
grace_hopper = np.array(grace_hopper)/255.0
grace_hopper.shape

In [ ]:
result = model.predict(grace_hopper[np.newaxis, ...])
result.shape

In [ ]:
predicted_class = np.argmax(result[0], axis=-1)
predicted_class

In [ ]:
labels_path = tf.keras.utils.get_file('ImageNetLabels.txt','https://storage.googleapis.com/download.tensorflow.org/data/ImageNetLabels.txt')
imagenet_labels = np.array(open(labels_path).read().splitlines())

plt.imshow(grace_hopper)
plt.axis('off')
predicted_class_name = imagenet_labels[predicted_class]
_ = plt.title("Prediction: " + predicted_class_name.title())

In [ ]:
(train_examples, validation_examples), info = tfds.load(
    'cats_vs_dogs',
    with_info=True,
    as_supervised=True,
    split=['train[:80%]', 'train[80%:]'],
)

num_examples = info.splits['train'].num_examples
num_classes = info.features['label'].num_classes

In [ ]:
for i, example_image in enumerate(train_examples.take(3)):
  print("Image {} shape: {}".format(i+1, example_image[0].shape))

So we need to reformat all images to the resolution expected by MobileNet (224, 224).

In [ ]:
def format_image(image, label):
  image = tf.image.resize(image, (IMAGE_RES, IMAGE_RES))/255.0
  return image, label

BATCH_SIZE = 32

train_batches      = train_examples.shuffle(num_examples//4).map(format_image).batch(BATCH_SIZE).prefetch(1)
validation_batches = validation_examples.map(format_image).batch(BATCH_SIZE).prefetch(1)

In [ ]:
image_batch, label_batch = next(iter(train_batches.take(1)))
image_batch = image_batch.numpy()
label_batch = label_batch.numpy()

result_batch = model.predict(image_batch)

predicted_class_names = imagenet_labels[np.argmax(result_batch, axis=-1)]
predicted_class_names

In [ ]:
plt.figure(figsize=(10,9))
for n in range(30):
  plt.subplot(6,5,n+1)
  plt.subplots_adjust(hspace = 0.3)
  plt.imshow(image_batch[n])
  plt.title(predicted_class_names[n])
  plt.axis('off')
_ = plt.suptitle("ImageNet predictions")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1. Load pretrained MobileNetV2 (without the top layer)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,   # remove default classification head
    weights='imagenet'   # use pretrained weights
)

# 2. Freeze the base model (we don't train these layers)
base_model.trainable = False

# 3. Build a Sequential model
model = models.Sequential([
    base_model,                            # pretrained feature extractor
    layers.GlobalAveragePooling2D(),       # reduce dimensions
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # binary classification (cat vs dog)
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

EPOCHS = 6
history = model.fit(train_batches,
                    epochs=EPOCHS,
                    validation_data=validation_batches)

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(EPOCHS)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

# EfficientNetB0

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

In [ ]:
# Load pretrained EfficientNetB0
base_model = EfficientNetB0(
    include_top=False,
    input_shape=(224, 224, 3),
    weights='imagenet'
)
base_model.trainable = False  # freeze pretrained layers

In [ ]:
# Build model
model = models.Sequential([
    layers.Rescaling(1./255 , input_shape=(224, 224, 3)),          # simple scaling instead of preprocess_input
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

In [ ]:
model.summary()

In [ ]:
# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
EPOCHS = 6
history = model.fit(train_batches,
                    epochs=EPOCHS,
                    validation_data=validation_batches)


In [ ]:
from tensorflow.keras.applications import ResNet50V2, EfficientNetB0, VGG16

# base_model = ResNet50V2(include_top=False, input_shape=(224,224,3), weights='imagenet')
# OR
# base_model = EfficientNetB0(include_top=False, input_shape=(224,224,3), weights='imagenet')
# OR
base_model = VGG16(include_top=False, input_shape=(224,224,3), weights='imagenet')


In [ ]:
base_model.trainable = False  # freeze pretrained layers

In [ ]:
# Build model
model = models.Sequential([
    layers.Rescaling(1./255 , input_shape=(224, 224, 3)),          # simple scaling instead of preprocess_input
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

In [ ]:
model.summary()

In [ ]:
# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
EPOCHS = 6
history = model.fit(train_batches,
                    epochs=EPOCHS,
                    validation_data=validation_batches)
